# E13 · MMMU-Pro 어댑터 A/B 평가
Colab GPU에서 셀을 위에서부터 실행합니다. 기본값은 Standard 10지선다 전체 평가입니다. 빠른 점검은 LIMIT를 3으로 바꾸세요. 두 어댑터는 같은 문제·프롬프트·이미지 해상도·생성 예산을 사용합니다. 학습 때 사용한 괄호 형식을 명시하는 팀 실험이며, 채점에는 공식 Pro 코드를 그대로 사용합니다.

In [3]:
%pip -q install "transformers==4.57.3" "peft==0.18.0" "accelerate>=1.2,<2" "datasets>=3,<5"

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast, hashlib, json, os, random, re, runpy, urllib.request
import pandas as pd
import torch, transformers, peft, datasets
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import PeftModel
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive')
ADAPTERS = {g: ROOT/'adapter'/f'adapter_{g}' for g in ('A', 'B')}
VARIANT = 'standard (10 options)'  # 또는 'vision', 'standard (4 options)'
LIMIT = None  # 점검할 때만 3
MAX_NEW = 2048
MAX_PIXELS = 602112
BASE = 'Qwen/Qwen3-VL-4B-Instruct'
BASE_REV = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'
DATA = 'MMMU/MMMU_Pro'
EVAL_REV = 'b26bdefc7e5e007962fde8f93b2bb53c99c8d13d'
PROMPT = "Answer the multiple-choice question. End with a line 'Answer: (X)' where X is your chosen option letter."
assert torch.cuda.is_available(), 'Colab 런타임을 GPU로 변경하세요.'
for folder in ADAPTERS.values():
    for filename in ('adapter_config.json', 'adapter_model.safetensors'):
        if not (folder/filename).is_file():
            raise FileNotFoundError(folder/filename)

Mounted at /content/drive


In [5]:
def digest_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(2**20), b''):
            h.update(block)
    return h.hexdigest()

data_rev = HfApi().dataset_info(DATA).sha
ds = load_dataset(DATA, VARIANT, split='test', revision=data_rev)
if LIMIT is not None:
    assert 0 < LIMIT <= len(ds)
    ds = ds.select(range(LIMIT))
ids = list(ds['id'])
assert len(ids) == len(set(ids))
meta = dict(variant=VARIANT, split='test', limit=LIMIT, count=len(ds), data=DATA,
            data_revision=data_rev, model=BASE, revision=BASE_REV, evaluator=EVAL_REV,
            max_new_tokens=MAX_NEW, max_pixels=MAX_PIXELS, prompt=PROMPT, decoding='greedy',
            parser_seed=42, transformers=transformers.__version__, peft=peft.__version__,
            torch=torch.__version__, gpu=torch.cuda.get_device_name(), runner_version=1,
            adapters={g: {n: digest_file(p/n) for n in ('adapter_config.json','adapter_model.safetensors')}
                      for g,p in ADAPTERS.items()})
key = hashlib.sha256(json.dumps(meta, sort_keys=True).encode()).hexdigest()[:16]
out = ROOT/'MMDL_E13_MMMU_Pro'/key
out.mkdir(parents=True, exist_ok=True)
(out/'config.json').write_text(json.dumps(meta, indent=2))
scorer = Path('/content')/f'mmmu_pro_eval_{EVAL_REV}.py'
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/MMMU-Benchmark/MMMU/{EVAL_REV}/mmmu-pro/evaluate.py', scorer)
score = runpy.run_path(str(scorer))['mmmu_process_results']
print(len(ds), '문항 × 2개 어댑터', out)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]

1730 문항 × 2개 어댑터 /content/drive/MyDrive/MMDL_E13_MMMU_Pro/1467073e950aeaab


In [6]:
def content_for(row, options):
    if VARIANT == 'vision':
        return [dict(type='image', image=row['image'].convert('RGB')), dict(type='text', text=PROMPT)]
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i,opt in enumerate(options))
    parts = re.split(r'(<image\s+\d+>)', row['question']+'\n'+choices+'\n'+PROMPT)
    content = []
    for part in parts:
        match = re.fullmatch(r'<image\s+(\d+)>', part)
        if match:
            content.append(dict(type='image', image=row['image_'+match[1]].convert('RGB')))
        elif part:
            content.append(dict(type='text', text=part))
    return content

processor = AutoProcessor.from_pretrained(BASE, revision=BASE_REV, max_pixels=MAX_PIXELS)
# Qwen3-VL 프로세서의 동적 해상도 상한을 실제 전처리 설정에도 반영합니다.
processor.image_processor.size['longest_edge'] = MAX_PIXELS
weight_key = (BASE, BASE_REV, json.dumps(meta['adapters'], sort_keys=True))
if globals().get('_loaded_weight_key') == weight_key and globals().get('model') is not None:
    print('메모리에 있는 기본 모델과 어댑터 A/B를 재사용합니다.')
else:
    import gc
    globals().pop('_loaded_weight_key', None)
    globals().pop('model', None)
    globals().pop('base_model', None)
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'{torch.cuda.get_device_name()} · 로딩 전 여유 {free/2**30:.2f} / {total/2**30:.2f} GiB')
    def load_eval_model():
        base = Qwen3VLForConditionalGeneration.from_pretrained(
            BASE, revision=BASE_REV, dtype=torch.bfloat16, device_map={'': 'cuda:0'},
            low_cpu_mem_usage=True)
        wrapped = PeftModel.from_pretrained(base, str(ADAPTERS['A']), adapter_name='A')
        wrapped.load_adapter(str(ADAPTERS['B']), adapter_name='B')
        return wrapped.eval()
    model = load_eval_model()
    _loaded_weight_key = weight_key

@torch.inference_mode()
def generate(row, options):
    messages = [dict(role='user', content=content_for(row, options))]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                            return_dict=True, return_tensors='pt').to(model.device)
    seq = model.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW)
    tail = seq[0, inputs.input_ids.shape[1]:]
    eos = model.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    capped = len(tail) == MAX_NEW and tail[-1].item() not in eos
    return processor.decode(tail, skip_special_tokens=True), len(tail), capped

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

... (6 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['astra_config', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [7]:
# 10문항 단위로 Drive에 저장합니다. 런타임이 끊기면 같은 설정으로 다시 실행하세요.
def save_text(path, text):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(text, encoding='utf-8')
    os.replace(tmp, path)

def lines(rows):
    return ''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in rows)

def evaluate_group(group):
    model.set_adapter(group)
    model.eval()
    folder = out/group
    folder.mkdir(exist_ok=True)
    rows = []
    for start in tqdm(range(0,len(ds),10), desc=group):
        stop = min(start+10,len(ds))
        path = folder/f'chunk_{start:05}.jsonl'
        if path.exists():
            batch = [json.loads(s) for s in path.read_text().splitlines()]
        else:
            batch = []
            for i in range(start,stop):
                row = ds[i]
                opts = ast.literal_eval(row['options']) if isinstance(row['options'],str) else row['options']
                response,n,capped = generate(row, opts)
                batch.append(dict(id=row['id'], options=opts, answer=row['answer'], response=response,
                                  generated_tokens=n, hit_limit=capped))
            save_text(path, lines(batch))
        assert [r['id'] for r in batch] == ids[start:stop], path
        rows.extend(batch)
    random.seed(42)  # 공식 파서의 fallback을 데이터 순서대로 재현
    for r in rows:
        score(r)
        last = r['response'].rstrip().split('\n')[-1]
        valid = ''.join(chr(65+i) for i in range(len(r['options'])))
        r['format_ok'] = bool(re.fullmatch(r'Answer: \(['+valid+r']\)',last))
    save_text(out/f'adapter_{group}.jsonl', lines(rows))
    df = pd.DataFrame(rows)
    return dict(adapter=group, questions=len(df), correct=int(df.if_right.sum()),
                accuracy_pct=100*df.if_right.mean(), format_pct=100*df.format_ok.mean(),
                hit_limit_pct=100*df.hit_limit.mean())

summary = pd.DataFrame([evaluate_group(g) for g in ADAPTERS])
save_text(out/'summary.csv',summary.to_csv(index=False))
display(summary.round(2))
print('저장 위치', out)

A:   0%|          | 0/173 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 6.69 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.26 GiB is free. Including non-PyTorch memory, this process has 11.30 GiB memory in use. Of the allocated memory 9.67 GiB is allocated by PyTorch, and 1.51 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 학습 전(어댑터 없음) 측정

어댑터를 끈 같은 모델을 같은 프롬프트로 돌려 기준선을 만듭니다.
시간 제약으로 3문항마다 1개(577문항)만 측정하고, 순서를 섞어(seed 42) 중간에 끊겨도 편향이 없게 했습니다.


In [ ]:
# 학습 전(어댑터 없음) 기준선. 셀 5와 같은 재개 방식(10문항 단위 Drive 저장)을 씁니다.
import json, random, re, os
from tqdm.auto import tqdm
import pandas as pd

def save_text(path, text):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(text, encoding='utf-8')
    os.replace(tmp, path)

def lines(rows):
    return ''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in rows)

STEP = 3
idx = list(range(0, len(ds), STEP))
random.Random(42).shuffle(idx)            # 끊겨도 편향 없게 섞음
sub = ds.select(idx)
sub_ids = list(sub['id'])

base_out = out.parent / (out.name + f'_base_step{STEP}')
base_out.mkdir(parents=True, exist_ok=True)
save_text(base_out/'subset_ids.json', json.dumps(sub_ids))

model.disable_adapter_layers(); model.eval()

rows = []
for start in tqdm(range(0, len(sub), 10), desc='base'):
    stop = min(start+10, len(sub))
    path = base_out/f'chunk_{start:05}.jsonl'
    if path.exists():
        batch = [json.loads(s) for s in path.read_text().splitlines()]
    else:
        batch = []
        for i in range(start, stop):
            row = sub[i]
            opts = ast.literal_eval(row['options']) if isinstance(row['options'], str) else row['options']
            response, n, capped = generate(row, opts)
            batch.append(dict(id=row['id'], options=opts, answer=row['answer'],
                              response=response, generated_tokens=n, hit_limit=capped))
        save_text(path, lines(batch))
    assert [r['id'] for r in batch] == sub_ids[start:stop], path
    rows.extend(batch)

random.seed(42)
for r in rows:
    score(r)
    last = r['response'].rstrip().split('\n')[-1]
    valid = ''.join(chr(65+i) for i in range(len(r['options'])))
    r['format_ok'] = bool(re.fullmatch(r'Answer: \(['+valid+r']\)', last))
save_text(base_out/'base.jsonl', lines(rows))

df = pd.DataFrame(rows)
print(f"학습 전 · {len(df)}문항 · 정확도 {100*df.if_right.mean():.2f}% · "
      f"형식 {100*df.format_ok.mean():.1f}% · 잘림 {100*df.hit_limit.mean():.1f}% · {base_out}")
model.enable_adapter_layers()


## 참고 · GPU를 바꿔 이어서 돌릴 때

출력 폴더 이름은 설정 메타의 SHA256이고, 메타에 `torch.cuda.get_device_name()`이 들어갑니다.
그래서 T4에서 L4로 바꾸면 폴더 이름이 달라지고 이미 돌린 청크를 다시 돌립니다.
이어받으려면 이전 폴더의 `chunk_*.jsonl`을 새 폴더로 복사한 뒤 셀 5를 실행하세요.

```python
import shutil
src = "/content/drive/MyDrive/MMDL_E13_MMMU_Pro/<이전_해시>/A"
shutil.copytree(src, out/'A', dirs_exist_ok=True)
print(len(list((out/'A').glob('chunk_*.jsonl'))), "개 복사됨")
```

이번 실험에서 A군 1,700문항이 T4(`1467073e950aeaab`)에서 측정됐고, 나머지 30문항과 B군·학습 전은 L4(`c1358a2778830849`)에서 측정됐습니다. greedy 디코딩이라 GPU 차이의 영향은 미미합니다.
